In [7]:
# Load results and reuse evaluate()
import duckdb
import numpy as np
import pandas as pd
import joblib

pipeline = joblib.load("../models/linear_model.joblib")
preds = pd.read_parquet("../data/gold/nyc/model_linear_predictions.parquet")

train_preds = preds[preds["split"] == "train"]
test_preds = preds[preds["split"] == "test"]

results = []

def evaluate(name, actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    missing = np.isnan(predicted)
    if missing.any():
        print(f"[{name}] WARNING: {missing.sum()} listings had no prediction")
    actual, predicted = actual[~missing], predicted[~missing]
    pct_error = np.abs(predicted - actual) / actual * 100
    row = {
        "model": name, "n_evaluated": len(actual),
        "median_abs_pct_error": round(np.median(pct_error), 1),
        "within_20pct": round((pct_error <= 20).mean() * 100, 1),
        "median_abs_error_$": round(np.median(np.abs(predicted - actual)), 1),
    }
    results.append(row)
    return row

evaluate("linear regression (train)", train_preds["base_price"], train_preds["predicted_base_price"])
evaluate("linear regression (test)", test_preds["base_price"], test_preds["predicted_base_price"])
pd.DataFrame(results)

,model,n_evaluated,median_abs_pct_error,within_20pct,median_abs_error_$
0,linear regression (train),17035,29.5,35.5,49.4
1,linear regression (test),4378,30.8,34.7,56.7


In [8]:
# Read the coefficients
model = pipeline.named_steps["model"]
preprocessing = pipeline.named_steps["preprocessing"]

# Get the name of every column the pipeline created
feature_names = preprocessing.get_feature_names_out()

coefficients = pd.DataFrame({
    "feature": feature_names,
    "coefficient": model.coef_,
}).sort_values("coefficient", ascending=False)

# Since the target is log10(price), each coefficient converts to
# a percentage change in price: 10^coefficient - 1
coefficients["approx_pct_change"] = (10 ** coefficients["coefficient"] - 1) * 100

print("Biggest positive effects on price:")
print(coefficients.head(10).round(3))
print()
print("Biggest negative effects on price:")
print(coefficients.tail(10).round(3))

Biggest positive effects on price:
                                     feature  coefficient  approx_pct_change
30            categorical__borough_Manhattan        0.143             39.001
16  numeric__missingindicator_minimum_nights        0.122             32.326
19    categorical__room_type_Entire home/apt        0.115             30.193
15       numeric__missingindicator_bathrooms        0.091             23.382
35        categorical__host_is_superhost_nan        0.090             22.998
18      numeric__missingindicator_host_years        0.090             22.998
17  numeric__missingindicator_rating_overall        0.087             22.252
31               categorical__borough_Queens        0.086             21.769
36            categorical__shared_bath_False        0.068             17.024
28                categorical__borough_Bronx        0.065             16.274

Biggest negative effects on price:
                                   feature  coefficient  approx_pct_change
23     

In [9]:
# Where does the model do worst?
test_merged = test_preds.merge(
    duckdb.sql(
        "SELECT listing_id, room_type, stay_type, borough "
        "FROM read_parquet('../data/gold/nyc/model_table.parquet')"
    ).df(),
    on="listing_id",
)
test_merged["pct_error"] = (
    (test_merged["predicted_base_price"] - test_merged["base_price"]).abs()
    / test_merged["base_price"] * 100
)

print(
    test_merged.groupby(["room_type", "stay_type"])["pct_error"]
    .agg(["count", "median"])
    .round(1)
    .sort_values("median", ascending=False)
)

                              count  median
room_type       stay_type                  
Hotel room      monthly stay     21    51.9
Shared room     monthly stay     23    48.4
                short stay       10    45.2
Entire home/apt short stay      385    39.5
Hotel room      short stay       89    35.3
Private room    monthly stay   1101    31.2
Entire home/apt monthly stay   2116    29.6
Private room    short stay      633    29.5


In [10]:
# # Check the raw values behind host_years and instant_bookable
check = duckdb.sql("""
    SELECT
        COUNT(*)                                   AS total,
        COUNT(host_since)                          AS has_host_since,
        COUNT(last_scraped)                        AS has_last_scraped,
        COUNT(instant_bookable)                    AS has_instant_bookable,
        MIN(host_since)                            AS earliest_host_since,
        MAX(host_since)                            AS latest_host_since,
        MIN(last_scraped)                          AS earliest_scraped,
        MAX(last_scraped)                          AS latest_scraped
    FROM read_parquet('../data/silver/nyc/listings.parquet')
""").df()
print(check.T)   # .T flips rows/columns so long labels are readable

                                        0
total                               30259
has_host_since                          0
has_last_scraped                    30259
has_instant_bookable                    0
earliest_host_since                   NaT
latest_host_since                     NaT
earliest_scraped      2026-06-14 00:00:00
latest_scraped        2026-06-23 00:00:00


In [11]:
# Look at the raw bronze text before any conversion
raw = duckdb.sql("""
    SELECT DISTINCT instant_bookable
    FROM read_parquet('../data/bronze/nyc/listings.parquet')
    LIMIT 10
""").df()
print(raw)

raw_dates = duckdb.sql("""
    SELECT host_since, last_scraped
    FROM read_parquet('../data/bronze/nyc/listings.parquet')
    LIMIT 5
""").df()
print(raw_dates)

  instant_bookable
0             None
  host_since last_scraped
0       None   2026-06-15
1       None   2026-06-14
2       None   2026-06-14
3       None   2026-06-14
4       None   2026-06-14


In [12]:
tenure = duckdb.sql("""
    SELECT
        COUNT(*)                                   AS total,
        COUNT(hosts_time_as_host_years)             AS has_host_years,
        COUNT(hosts_time_as_host_months)            AS has_host_months,
        MIN(TRY_CAST(hosts_time_as_host_years AS DOUBLE)) AS min_years,
        MAX(TRY_CAST(hosts_time_as_host_years AS DOUBLE)) AS max_years
    FROM read_parquet('../data/bronze/nyc/listings.parquet')
""").df()
print(tenure.T)

sample = duckdb.sql("""
    SELECT hosts_time_as_host_years, hosts_time_as_host_months
    FROM read_parquet('../data/bronze/nyc/listings.parquet')
    WHERE hosts_time_as_host_years IS NOT NULL
    LIMIT 5
""").df()
print(sample)

                       0
total            30259.0
has_host_years   29910.0
has_host_months  29910.0
min_years            0.0
max_years           15.0
  hosts_time_as_host_years hosts_time_as_host_months
0                       15                         5
1                       15                         5
2                       15                         5
3                       15                         5
4                       15                         5


In [13]:
tenure = duckdb.sql("""
    SELECT
        COUNT(*)                                   AS total,
        COUNT(host_years)                          AS has_host_years,
        MIN(host_years)                            AS min_years,
        MAX(host_years)                            AS max_years
    FROM read_parquet('../data/gold/nyc/model_table.parquet')
""").df()
print(tenure.T)

                      0
total           21514.0
has_host_years  21236.0
min_years           0.0
max_years          15.0


In [14]:
distribution = duckdb.sql("""
    SELECT
        TRY_CAST(hosts_time_as_host_years AS DOUBLE) AS years,
        COUNT(*) AS listings
    FROM read_parquet('../data/bronze/nyc/listings.parquet')
    WHERE hosts_time_as_host_years IS NOT NULL
    GROUP BY 1
    ORDER BY 1
""").df()
print(distribution.to_string(index=False))
print()
print("Share at the max (15 years):",
      round(100 * (distribution.loc[distribution["years"] == 15, "listings"].sum()
                    / distribution["listings"].sum()), 1), "%")

 years  listings
   0.0      1420
   1.0      1254
   2.0      1886
   3.0      3871
   4.0      2547
   5.0      1485
   6.0      1894
   7.0      2206
   8.0      2208
   9.0      2557
  10.0      3001
  11.0      2320
  12.0      1260
  13.0       912
  14.0       649
  15.0       440

Share at the max (15 years): 1.5 %
